In [1]:
import yfinance as yf
import pandas as pd

In [2]:
tickers = {
    "AI.PA": "Air Liquide",
    "AIR.PA": "Airbus",
    "CAP.PA": "Capgemini",
    "DG.PA": "Vinci",
    "MC.PA": "LVMH",
    "OR.PA": "L'Oréal",
    "SAN.PA": "Sanofi",
    "TTE.PA": "TotalEnergies",
    "BNP.PA": "BNP Paribas",
    "CS.PA": "AXA",
    "RI.PA": "Pernod Ricard",
    "SU.PA": "Schneider Electric",
    "EN.PA": "Bouygues",
    "DSY.PA": "Dassault Systèmes",
    "HO.PA": "Thales"
}

secteurs_haram = ["Financial Services"]
industries_haram = [
    "Alcoholic Beverages", "Breweries", "Wineries",
    "Distillers & Vintners", "Tobacco", "Gambling",
    "Adult Entertainment", "Weapons & Ammunition"
]

In [3]:
resultats = []

for ticker, nom in tickers.items():
    try:
        info = yf.Ticker(ticker).info
        sector = info.get('sector', 'Inconnu')
        industry = info.get('industry', 'Inconnu')
        market_cap = info.get('marketCap', 0)
        total_debt = info.get('totalDebt', 0)

        if sector in secteurs_haram or industry in industries_haram:
            statut = "HARAM"
            ratio = "N/A"
        else:
            if market_cap > 0:
                ratio = round(total_debt / market_cap * 100, 1)
            else:
                ratio = 0

            if ratio > 33:
                statut = "NON CONFORME"
            else:
                statut = "CONFORME"

        resultats.append({
            "Nom": nom,
            "Secteur": sector,
            "Industrie": industry,
            "Dette/Cap (%)": ratio,
            "Statut": statut
        })
        print(f"✅ {nom} analysé")

    except Exception as e:
        print(f"❌ Erreur {nom}: {e}")

✅ Air Liquide analysé
✅ Airbus analysé
✅ Capgemini analysé
✅ Vinci analysé
✅ LVMH analysé
✅ L'Oréal analysé
✅ Sanofi analysé
✅ TotalEnergies analysé
✅ BNP Paribas analysé
✅ AXA analysé
✅ Pernod Ricard analysé
✅ Schneider Electric analysé
✅ Bouygues analysé
✅ Dassault Systèmes analysé
✅ Thales analysé


In [4]:
df = pd.DataFrame(resultats)

print("\n" + "=" * 60)
print("  SCREENER SHARIA — RÉSULTATS CAC 40")
print("=" * 60)

conformes = df[df['Statut'] == 'CONFORME']
print(f"\n✅ CONFORMES ({len(conformes)}):")
print(conformes[['Nom', 'Dette/Cap (%)', 'Secteur']].to_string(index=False))

non_conformes = df[df['Statut'] == 'NON CONFORME']
print(f"\n⚠️ NON CONFORMES - dette > 33% ({len(non_conformes)}):")
print(non_conformes[['Nom', 'Dette/Cap (%)', 'Secteur']].to_string(index=False))

haram = df[df['Statut'] == 'HARAM']
print(f"\n❌ HARAM - secteur interdit ({len(haram)}):")
print(haram[['Nom', 'Industrie']].to_string(index=False))


  SCREENER SHARIA — RÉSULTATS CAC 40

✅ CONFORMES (9):
               Nom Dette/Cap (%)            Secteur
       Air Liquide          14.0    Basic Materials
            Airbus          10.5        Industrials
         Capgemini           0.0         Technology
              LVMH          15.0  Consumer Cyclical
           L'Oréal           6.2 Consumer Defensive
            Sanofi          22.0         Healthcare
Schneider Electric          14.0        Industrials
 Dassault Systèmes          10.6         Technology
            Thales          11.7        Industrials

⚠️ NON CONFORMES - dette > 33% (4):
          Nom Dette/Cap (%)            Secteur
        Vinci          56.7        Industrials
TotalEnergies          41.5             Energy
Pernod Ricard          77.6 Consumer Defensive
     Bouygues          80.1        Industrials

❌ HARAM - secteur interdit (2):
        Nom               Industrie
BNP Paribas        Banks - Regional
        AXA Insurance - Diversified


In [5]:
import yfinance as yf
import pandas as pd

tickers = {
    "AI.PA": "Air Liquide",
    "AIR.PA": "Airbus",
    "CAP.PA": "Capgemini",
    "DG.PA": "Vinci",
    "MC.PA": "LVMH",
    "OR.PA": "L'Oréal",
    "SAN.PA": "Sanofi",
    "TTE.PA": "TotalEnergies",
    "BNP.PA": "BNP Paribas",
    "CS.PA": "AXA",
    "RI.PA": "Pernod Ricard",
    "SU.PA": "Schneider Electric",
    "EN.PA": "Bouygues",
    "DSY.PA": "Dassault Systèmes",
    "HO.PA": "Thales"
}

secteurs_haram = ["Financial Services"]
industries_haram = [
    "Alcoholic Beverages", "Breweries", "Wineries",
    "Distillers & Vintners", "Tobacco", "Gambling",
    "Adult Entertainment", "Weapons & Ammunition"
]

In [6]:
resultats_v2 = []

for ticker, nom in tickers.items():
    try:
        info = yf.Ticker(ticker).info
        sector = info.get('sector', 'Inconnu')
        industry = info.get('industry', 'Inconnu')
        market_cap = info.get('marketCap', 0)
        total_debt = info.get('totalDebt', 0)
        total_cash = info.get('totalCash', 0)
        total_revenue = info.get('totalRevenue', 0)
        receivables = info.get('netReceivables', 0)

        # Filtre sectoriel
        if sector in secteurs_haram or industry in industries_haram:
            statut = "HARAM"
            r_dette = "N/A"
            r_cash = "N/A"
            r_creances = "N/A"
            score = 0
        else:
            # Ratio 1 : Dette / Capitalisation (seuil < 33%)
            r_dette = round(total_debt / market_cap * 100, 1) if market_cap > 0 else 0

            # Ratio 2 : Cash + placements / Capitalisation (seuil < 33%)
            r_cash = round(total_cash / market_cap * 100, 1) if market_cap > 0 else 0

            # Ratio 3 : Créances / Capitalisation (seuil < 49%)
            r_creances = round(receivables / market_cap * 100, 1) if market_cap > 0 else 0

            # Statut final
            if r_dette > 33:
                statut = "NON CONFORME (dette)"
            elif r_cash > 33:
                statut = "NON CONFORME (cash)"
            elif r_creances > 49:
                statut = "NON CONFORME (créances)"
            else:
                statut = "CONFORME"

            # Score de conformité (0-100)
            score_dette = max(0, 100 - (r_dette / 33 * 100))
            score_cash = max(0, 100 - (r_cash / 33 * 100))
            score_creances = max(0, 100 - (r_creances / 49 * 100))
            score = round((score_dette + score_cash + score_creances) / 3, 0)

        resultats_v2.append({
            "Nom": nom,
            "Secteur": sector,
            "Dette/Cap (%)": r_dette,
            "Cash/Cap (%)": r_cash,
            "Créances/Cap (%)": r_creances,
            "Score": score,
            "Statut": statut
        })
        print(f"✅ {nom} — Score: {score}")

    except Exception as e:
        print(f"❌ Erreur {nom}: {e}")

✅ Air Liquide — Score: 82.0
✅ Airbus — Score: 76.0
✅ Capgemini — Score: 100.0
✅ Vinci — Score: 41.0
✅ LVMH — Score: 79.0
✅ L'Oréal — Score: 89.0
✅ Sanofi — Score: 69.0
✅ TotalEnergies — Score: 47.0
✅ BNP Paribas — Score: 0
✅ AXA — Score: 0
✅ Pernod Ricard — Score: 53.0
✅ Schneider Electric — Score: 83.0
✅ Bouygues — Score: 33.0
✅ Dassault Systèmes — Score: 72.0
✅ Thales — Score: 79.0


In [7]:
df2 = pd.DataFrame(resultats_v2)

conformes = df2[df2['Statut'] == 'CONFORME'].sort_values('Score', ascending=False)

print("=" * 70)
print("  SCREENER SHARIA V2 — CAC 40 (3 ratios AAOIFI + Score)")
print("=" * 70)
print(f"\n✅ ACTIONS CONFORMES — classées par score ({len(conformes)}):\n")
print(conformes[['Nom', 'Score', 'Dette/Cap (%)', 'Cash/Cap (%)', 'Créances/Cap (%)']].to_string(index=False))

non_conf = df2[df2['Statut'].str.contains('NON CONFORME')]
print(f"\n⚠️ NON CONFORMES ({len(non_conf)}):\n")
print(non_conf[['Nom', 'Statut', 'Dette/Cap (%)', 'Cash/Cap (%)', 'Créances/Cap (%)']].to_string(index=False))

haram = df2[df2['Statut'] == 'HARAM']
print(f"\n❌ HARAM ({len(haram)}):\n")
print(haram[['Nom', 'Secteur']].to_string(index=False))

  SCREENER SHARIA V2 — CAC 40 (3 ratios AAOIFI + Score)

✅ ACTIONS CONFORMES — classées par score (9):

               Nom  Score Dette/Cap (%) Cash/Cap (%) Créances/Cap (%)
         Capgemini  100.0           0.0          0.0              0.0
           L'Oréal   89.0           6.2          5.1              0.0
Schneider Electric   83.0          14.0          3.3              0.0
       Air Liquide   82.0          14.0          4.1              0.0
              LVMH   79.0          15.0          5.5              0.0
            Thales   79.0          11.7          8.6              0.0
            Airbus   76.0          10.5         13.3              0.0
 Dassault Systèmes   72.0          10.6         16.8              0.0
            Sanofi   69.0          22.0          8.3              0.0

⚠️ NON CONFORMES (4):

          Nom               Statut Dette/Cap (%) Cash/Cap (%) Créances/Cap (%)
        Vinci NON CONFORME (dette)          56.7         25.7              0.0
TotalEnergies 

In [8]:
liste_noire = {
    "MC.PA": "LVMH - branche alcool (Moët Hennessy) > 5% du CA",
    "RI.PA": "Pernod Ricard - activité principale = alcool",
}

print("🔍 VÉRIFICATION LISTE NOIRE :\n")
for ticker, raison in liste_noire.items():
    nom = tickers.get(ticker, ticker)
    print(f"⚠️  {nom} → RETIRÉ — Raison : {raison}")

df3 = df2[~df2['Nom'].isin(["LVMH", "Pernod Ricard"])]
conformes_final = df3[df3['Statut'] == 'CONFORME'].sort_values('Score', ascending=False)

print(f"\n✅ ACTIONS CONFORMES FINALES ({len(conformes_final)}):\n")
print(conformes_final[['Nom', 'Score', 'Dette/Cap (%)', 'Cash/Cap (%)', 'Créances/Cap (%)']].to_string(index=False))

🔍 VÉRIFICATION LISTE NOIRE :

⚠️  LVMH → RETIRÉ — Raison : LVMH - branche alcool (Moët Hennessy) > 5% du CA
⚠️  Pernod Ricard → RETIRÉ — Raison : Pernod Ricard - activité principale = alcool

✅ ACTIONS CONFORMES FINALES (8):

               Nom  Score Dette/Cap (%) Cash/Cap (%) Créances/Cap (%)
         Capgemini  100.0           0.0          0.0              0.0
           L'Oréal   89.0           6.2          5.1              0.0
Schneider Electric   83.0          14.0          3.3              0.0
       Air Liquide   82.0          14.0          4.1              0.0
            Thales   79.0          11.7          8.6              0.0
            Airbus   76.0          10.5         13.3              0.0
 Dassault Systèmes   72.0          10.6         16.8              0.0
            Sanofi   69.0          22.0          8.3              0.0


In [9]:
resultats_v4 = []

for ticker, nom in tickers.items():
    try:
        stock = yf.Ticker(ticker)
        info = stock.info

        sector = info.get('sector', 'Inconnu')
        industry = info.get('industry', 'Inconnu')
        market_cap = info.get('marketCap', 0)
        total_debt = info.get('totalDebt', 0)
        total_cash = info.get('totalCash', 0)
        receivables = info.get('netReceivables', 0)
        
        # Données financières supplémentaires
        prix = info.get('currentPrice', 0)
        per = info.get('trailingPE', 'N/A')
        dividend_yield = info.get('dividendYield', 0)
        roe = info.get('returnOnEquity', 0)
        marge = info.get('profitMargins', 0)
        ca_growth = info.get('revenueGrowth', 0)
        beta = info.get('beta', 'N/A')

        # ── FILTRE 1 : Secteur haram ──
        if sector in secteurs_haram or industry in industries_haram:
            statut = "HARAM"
            r_dette = "N/A"
            r_cash = "N/A"
            r_creances = "N/A"
            score = 0
            purification = 0
        
        # ── FILTRE 2 : Liste noire (revenus haram > 5%) ──
        elif ticker in liste_noire:
            statut = "HARAM (revenus)"
            r_dette = "N/A"
            r_cash = "N/A"
            r_creances = "N/A"
            score = 0
            purification = 0
        
        else:
            # ── FILTRE 3 : Ratios AAOIFI ──
            r_dette = round(total_debt / market_cap * 100, 1) if market_cap > 0 else 0
            r_cash = round(total_cash / market_cap * 100, 1) if market_cap > 0 else 0
            r_creances = round(receivables / market_cap * 100, 1) if market_cap > 0 else 0

            if r_dette > 33:
                statut = "NON CONFORME (dette)"
            elif r_cash > 33:
                statut = "NON CONFORME (cash)"
            elif r_creances > 49:
                statut = "NON CONFORME (créances)"
            else:
                statut = "CONFORME"

            # Score de conformité
            score_dette = max(0, 100 - (r_dette / 33 * 100))
            score_cash = max(0, 100 - (r_cash / 33 * 100))
            score_creances = max(0, 100 - (r_creances / 49 * 100))
            score = round((score_dette + score_cash + score_creances) / 3, 0)

            # Taux de purification estimé (basé sur les revenus d'intérêts)
            interest_income = info.get('interestIncome', 0)
            total_revenue = info.get('totalRevenue', 1)
            if interest_income and total_revenue:
                purification = round(interest_income / total_revenue * 100, 2)
            else:
                purification = 0

        resultats_v4.append({
            "Ticker": ticker,
            "Nom": nom,
            "Secteur": sector,
            "Prix (€)": round(prix, 2) if prix else 0,
            "PER": round(per, 1) if isinstance(per, (int, float)) else "N/A",
            "Dividende (%)": round(dividend_yield * 100, 2) if dividend_yield else 0,
            "ROE (%)": round(roe * 100, 1) if roe else 0,
            "Marge nette (%)": round(marge * 100, 1) if marge else 0,
            "Croissance CA (%)": round(ca_growth * 100, 1) if ca_growth else 0,
            "Beta": round(beta, 2) if isinstance(beta, (int, float)) else "N/A",
            "Dette/Cap (%)": r_dette,
            "Cash/Cap (%)": r_cash,
            "Créances/Cap (%)": r_creances,
            "Score Sharia": score,
            "Purification (%)": purification,
            "Statut": statut
        })
        print(f"✅ {nom} — Score: {score} — Prix: {round(prix, 2)}€")

    except Exception as e:
        print(f"❌ Erreur {nom}: {e}")

print("\n✅ Analyse terminée !")

✅ Air Liquide — Score: 82.0 — Prix: 169.12€
✅ Airbus — Score: 76.0 — Prix: 171.34€
✅ Capgemini — Score: 100.0 — Prix: 109.5€
✅ Vinci — Score: 41.0 — Prix: 129.6€
✅ LVMH — Score: 0 — Prix: 494.15€
✅ L'Oréal — Score: 89.0 — Prix: 361.15€
✅ Sanofi — Score: 69.0 — Prix: 76.42€
✅ TotalEnergies — Score: 47.0 — Prix: 69.76€
✅ BNP Paribas — Score: 0 — Prix: 86.32€
✅ AXA — Score: 0 — Prix: 37.96€
✅ Pernod Ricard — Score: 0 — Prix: 69.6€
✅ Schneider Electric — Score: 83.0 — Prix: 252.25€
✅ Bouygues — Score: 33.0 — Prix: 49.32€
✅ Dassault Systèmes — Score: 72.0 — Prix: 18.61€
✅ Thales — Score: 79.0 — Prix: 255.1€

✅ Analyse terminée !


In [10]:
df4 = pd.DataFrame(resultats_v4)

# ── Actions conformes ──
conf = df4[df4['Statut'] == 'CONFORME'].sort_values('Score Sharia', ascending=False)

print("=" * 90)
print("  SCREENER SHARIA V4 — CAC 40 (Analyse complète)")
print("=" * 90)

print(f"\n✅ ACTIONS CONFORMES ({len(conf)}):\n")
print(conf[['Nom', 'Score Sharia', 'Prix (€)', 'PER', 'Dividende (%)', 
            'ROE (%)', 'Marge nette (%)', 'Croissance CA (%)', 'Beta',
            'Dette/Cap (%)', 'Purification (%)']].to_string(index=False))

# ── Non conformes ──
non_conf = df4[df4['Statut'].str.contains('NON CONFORME')]
print(f"\n⚠️ NON CONFORMES ({len(non_conf)}):\n")
print(non_conf[['Nom', 'Statut', 'Prix (€)', 'Dette/Cap (%)']].to_string(index=False))

# ── Haram ──
haram = df4[df4['Statut'].str.contains('HARAM')]
print(f"\n❌ HARAM ({len(haram)}):\n")
print(haram[['Nom', 'Statut', 'Secteur']].to_string(index=False))

# ── Résumé ──
print(f"\n{'=' * 90}")
print(f"📊 RÉSUMÉ : {len(conf)} conformes | {len(non_conf)} non conformes | {len(haram)} haram | sur {len(df4)} analysées")
print(f"{'=' * 90}")

  SCREENER SHARIA V4 — CAC 40 (Analyse complète)

✅ ACTIONS CONFORMES (8):

               Nom  Score Sharia  Prix (€)  PER  Dividende (%)  ROE (%)  Marge nette (%)  Croissance CA (%)  Beta Dette/Cap (%)  Purification (%)
         Capgemini         100.0    109.50 12.0          316.0      0.0              7.1                3.7  0.67           0.0                 0
           L'Oréal          89.0    361.15 31.6          199.0     18.0             13.9                1.0  0.88           6.2                 0
Schneider Electric          83.0    252.25 31.7          164.0     15.6             10.4                4.2  1.03          14.0                 0
       Air Liquide          82.0    169.12 27.7          221.0     13.4             13.1               -3.4  0.72          14.0                 0
            Thales          79.0    255.10 31.4          158.0     21.1              7.6                7.1  0.13          11.7                 0
            Airbus          76.0    171.34 26.0 

In [11]:
# CAC 40 complet (40 actions)
tickers_complet = {
    "AI.PA": "Air Liquide",
    "AIR.PA": "Airbus",
    "ALO.PA": "Schneider Electric",
    "MT.AS": "ArcelorMittal",
    "CS.PA": "AXA",
    "BNP.PA": "BNP Paribas",
    "EN.PA": "Bouygues",
    "CAP.PA": "Capgemini",
    "CA.PA": "Carrefour",
    "ACA.PA": "Crédit Agricole",
    "BN.PA": "Danone",
    "DSY.PA": "Dassault Systèmes",
    "ENGI.PA": "Engie",
    "EL.PA": "EssilorLuxottica",
    "ERF.PA": "Eurofins Scientific",
    "RMS.PA": "Hermès",
    "KER.PA": "Kering",
    "LR.PA": "Legrand",
    "OR.PA": "L'Oréal",
    "MC.PA": "LVMH",
    "ML.PA": "Michelin",
    "ORA.PA": "Orange",
    "RI.PA": "Pernod Ricard",
    "PUB.PA": "Publicis",
    "RNO.PA": "Renault",
    "SAF.PA": "Safran",
    "SGO.PA": "Saint-Gobain",
    "SAN.PA": "Sanofi",
    "SU.PA": "Schneider Electric",
    "GLE.PA": "Société Générale",
    "STLAP.PA": "Stellantis",
    "STMPA.PA": "STMicroelectronics",
    "TEP.PA": "Teleperformance",
    "HO.PA": "Thales",
    "TTE.PA": "TotalEnergies",
    "URW.AS": "Unibail-Rodamco",
    "VIE.PA": "Veolia",
    "DG.PA": "Vinci",
    "VIV.PA": "Vivendi",
    "WLN.PA": "Worldline"
}

# Liste noire étendue
liste_noire_v2 = {
    "MC.PA": "Branche alcool (Moët Hennessy) > 5% du CA",
    "RI.PA": "Activité principale = alcool",
    "KER.PA": "Kering possède des vignobles (Château Latour)",
}

print(f"Lancement de l'analyse de {len(tickers_complet)} actions...\n")

resultats_cac40 = []

for ticker, nom in tickers_complet.items():
    try:
        info = yf.Ticker(ticker).info
        sector = info.get('sector', 'Inconnu')
        industry = info.get('industry', 'Inconnu')
        market_cap = info.get('marketCap', 0)
        total_debt = info.get('totalDebt', 0)
        total_cash = info.get('totalCash', 0)
        receivables = info.get('netReceivables', 0)
        prix = info.get('currentPrice', 0)
        per = info.get('trailingPE', 'N/A')
        dividend_yield = info.get('dividendYield', 0)
        roe = info.get('returnOnEquity', 0)
        marge = info.get('profitMargins', 0)
        ca_growth = info.get('revenueGrowth', 0)
        beta = info.get('beta', 'N/A')

        # FILTRE 1 : Secteur
        if sector in secteurs_haram or industry in industries_haram:
            statut = "HARAM (secteur)"
            r_dette, r_cash, r_creances, score, purification = "N/A", "N/A", "N/A", 0, 0

        # FILTRE 2 : Liste noire
        elif ticker in liste_noire_v2:
            statut = "HARAM (revenus)"
            r_dette, r_cash, r_creances, score, purification = "N/A", "N/A", "N/A", 0, 0

        # FILTRE 3 : Ratios AAOIFI
        else:
            r_dette = round(total_debt / market_cap * 100, 1) if market_cap > 0 else 0
            r_cash = round(total_cash / market_cap * 100, 1) if market_cap > 0 else 0
            r_creances = round(receivables / market_cap * 100, 1) if market_cap > 0 else 0

            if r_dette > 33:
                statut = "NON CONFORME (dette)"
            elif r_cash > 33:
                statut = "NON CONFORME (cash)"
            elif r_creances > 49:
                statut = "NON CONFORME (créances)"
            else:
                statut = "CONFORME"

            score_dette = max(0, 100 - (r_dette / 33 * 100))
            score_cash = max(0, 100 - (r_cash / 33 * 100))
            score_creances = max(0, 100 - (r_creances / 49 * 100))
            score = round((score_dette + score_cash + score_creances) / 3, 0)

            interest_income = info.get('interestIncome', 0)
            total_revenue = info.get('totalRevenue', 1)
            if interest_income and total_revenue:
                purification = round(interest_income / total_revenue * 100, 2)
            else:
                purification = 0

        resultats_cac40.append({
            "Ticker": ticker,
            "Nom": nom,
            "Secteur": sector,
            "Prix (€)": round(prix, 2) if prix else 0,
            "PER": round(per, 1) if isinstance(per, (int, float)) else "N/A",
            "Dividende (%)": round(dividend_yield * 100, 2) if dividend_yield else 0,
            "ROE (%)": round(roe * 100, 1) if roe else 0,
            "Marge nette (%)": round(marge * 100, 1) if marge else 0,
            "Croissance CA (%)": round(ca_growth * 100, 1) if ca_growth else 0,
            "Beta": round(beta, 2) if isinstance(beta, (int, float)) else "N/A",
            "Dette/Cap (%)": r_dette,
            "Cash/Cap (%)": r_cash,
            "Créances/Cap (%)": r_creances,
            "Score Sharia": score,
            "Purification (%)": purification,
            "Statut": statut
        })
        print(f"✅ {nom}")

    except Exception as e:
        print(f"❌ {nom}: {e}")

print(f"\n✅ Analyse terminée — {len(resultats_cac40)} actions traitées")

Lancement de l'analyse de 40 actions...

✅ Air Liquide
✅ Airbus
✅ Schneider Electric
✅ ArcelorMittal
✅ AXA
✅ BNP Paribas
✅ Bouygues
✅ Capgemini
✅ Carrefour
✅ Crédit Agricole
✅ Danone
✅ Dassault Systèmes
✅ Engie
✅ EssilorLuxottica
✅ Eurofins Scientific
✅ Hermès
✅ Kering
✅ Legrand
✅ L'Oréal
✅ LVMH
✅ Michelin
✅ Orange
✅ Pernod Ricard
✅ Publicis
✅ Renault
✅ Safran
✅ Saint-Gobain
✅ Sanofi
✅ Schneider Electric
✅ Société Générale
✅ Stellantis
✅ STMicroelectronics
✅ Teleperformance
✅ Thales
✅ TotalEnergies
✅ Unibail-Rodamco
✅ Veolia
✅ Vinci
✅ Vivendi
✅ Worldline

✅ Analyse terminée — 40 actions traitées


In [12]:
df_cac40 = pd.DataFrame(resultats_cac40)

conf = df_cac40[df_cac40['Statut'] == 'CONFORME'].sort_values('Score Sharia', ascending=False)
non_conf = df_cac40[df_cac40['Statut'].str.contains('NON CONFORME')]
haram = df_cac40[df_cac40['Statut'].str.contains('HARAM')]

print("=" * 90)
print("  HALAL QUANT — SCREENER CAC 40 COMPLET (40 actions)")
print("=" * 90)

print(f"\n✅ ACTIONS CONFORMES ({len(conf)}):\n")
print(conf[['Nom', 'Score Sharia', 'Prix (€)', 'PER', 'Dividende (%)',
            'ROE (%)', 'Marge nette (%)', 'Beta', 'Dette/Cap (%)',
            'Purification (%)']].to_string(index=False))

print(f"\n⚠️ NON CONFORMES ({len(non_conf)}):\n")
print(non_conf[['Nom', 'Statut', 'Prix (€)', 'Dette/Cap (%)',
                'Cash/Cap (%)', 'Créances/Cap (%)']].to_string(index=False))

print(f"\n❌ HARAM ({len(haram)}):\n")
print(haram[['Nom', 'Statut', 'Secteur']].to_string(index=False))

print(f"\n{'=' * 90}")
print(f"📊 RÉSUMÉ CAC 40 : {len(conf)} conformes | {len(non_conf)} non conformes | {len(haram)} haram")
print(f"📈 Taux de conformité : {round(len(conf)/len(df_cac40)*100)}%")
print(f"{'=' * 90}")

  HALAL QUANT — SCREENER CAC 40 COMPLET (40 actions)

✅ ACTIONS CONFORMES (16):

               Nom  Score Sharia  Prix (€)    PER  Dividende (%)  ROE (%)  Marge nette (%)  Beta Dette/Cap (%)  Purification (%)
         Capgemini         100.0    109.50   12.0          316.0      0.0              7.1  0.67           0.0                 0
          Michelin         100.0     29.77    2.9            0.0      0.0              0.0   N/A           0.0                 0
   Unibail-Rodamco         100.0      0.00    N/A            0.0      0.0              0.0   N/A             0                 0
            Hermès          93.0   1908.00   44.4           94.0     25.2             28.3  0.94           1.2                 0
            Safran          90.0    306.20   17.8          105.0     55.6             23.0  0.92           4.2                 0
           L'Oréal          89.0    361.20   31.6          199.0     18.0             13.9  0.88           6.2                 0
Schneider Electr

In [13]:
# ══════════════════════════════════════════════════════════════
# HALAL QUANT V5 FINAL — SCREENING SHARIA MAXIMUM
# ══════════════════════════════════════════════════════════════

# ── SECTEURS INTERDITS (niveau 1) ──
SECTEURS_HARAM = [
    "Financial Services",
]

# ── INDUSTRIES INTERDITES (niveau 2 - très détaillé) ──
INDUSTRIES_HARAM = [
    # Banques
    "Banks - Regional", "Banks - Diversified", "Banks - Global",
    # Assurances conventionnelles
    "Insurance - Diversified", "Insurance - Life", 
    "Insurance - Property & Casualty", "Insurance - Reinsurance",
    "Insurance - Specialty", "Insurance Brokers",
    # Crédit et prêts
    "Credit Services", "Mortgage Finance", "Consumer Lending",
    # Finance de marché conventionnelle
    "Capital Markets", "Financial Data & Stock Exchanges",
    "Asset Management", "Investment Banking & Brokerage",
    # Alcool
    "Alcoholic Beverages", "Breweries", "Wineries", 
    "Distillers & Vintners",
    # Tabac
    "Tobacco", "Tobacco Products",
    # Jeux d'argent
    "Gambling", "Casinos & Gaming", "Sports Betting",
    "Lottery Services", "Online Gambling",
    # Armement
    "Weapons & Ammunition",
    # Divertissement adulte
    "Adult Entertainment",
]

# ── INDUSTRIES À VÉRIFIER MANUELLEMENT (zone grise) ──
INDUSTRIES_ATTENTION = {
    "Aerospace & Defense": "Vérifier si > 5% du CA vient de l'armement offensif",
    "Restaurants": "Vérifier si l'entreprise sert de l'alcool ou du porc",
    "Packaged Foods": "Vérifier si produits à base de porc ou gélatine porcine",
    "Grocery Stores": "Vérifier le % de CA provenant de l'alcool et du porc",
    "Hotels & Resorts": "Vérifier présence de bars, casinos, minibar alcool",
    "Entertainment": "Vérifier la nature du contenu (musique, films, etc.)",
    "Drug Manufacturers": "Vérifier si utilisation de gélatine porcine ou alcool",
    "Biotechnology": "Vérifier si recherche sur cellules souches embryonnaires",
    "Food Distribution": "Vérifier le % de distribution d'alcool et porc",
}

# ── LISTE NOIRE (vérification manuelle confirmée) ──
LISTE_NOIRE = {
    "MC.PA": "LVMH — Moët Hennessy (alcool) représente ~8% du CA",
    "RI.PA": "Pernod Ricard — activité principale = production d'alcool (100%)",
    "KER.PA": "Kering — possède Château Latour (vignobles, production de vin)",
}

# ── SEUILS AAOIFI ──
SEUIL_DETTE = 33
SEUIL_CASH = 33
SEUIL_CREANCES = 49
SEUIL_REV_HARAM = 5

print("=" * 80)
print("  HALAL QUANT V5 FINAL — SCREENING SHARIA COMPLET")
print("-" * 80)
print(f"  Norme : AAOIFI")
print(f"  Seuils : Dette < {SEUIL_DETTE}% | Cash < {SEUIL_CASH}% | "
      f"Créances < {SEUIL_CREANCES}% | Revenus haram < {SEUIL_REV_HARAM}%")
print(f"  Filtres : Sectoriel + Industrie + Liste noire + "
      f"Revenus d'intérêts + 3 ratios AAOIFI")
print("=" * 80)

resultats_final = []

for ticker, nom in tickers_complet.items():
    try:
        info = yf.Ticker(ticker).info
        sector = info.get('sector', 'Inconnu')
        industry = info.get('industry', 'Inconnu')
        market_cap = info.get('marketCap', 0)
        total_debt = info.get('totalDebt', 0)
        total_cash = info.get('totalCash', 0)
        receivables = info.get('netReceivables', 0)
        prix = info.get('currentPrice', 0)
        per = info.get('trailingPE', 'N/A')
        dividend_yield = info.get('dividendYield', 0)
        roe = info.get('returnOnEquity', 0)
        marge = info.get('profitMargins', 0)
        ca_growth = info.get('revenueGrowth', 0)
        beta = info.get('beta', 'N/A')
        interest_income = info.get('interestIncome', 0)
        interest_expense = info.get('interestExpense', 0)
        total_revenue = info.get('totalRevenue', 1)

        attention = ""
        raison = ""

        # ══════ NIVEAU 1 : Secteur principal ══════
        if sector in SECTEURS_HARAM:
            statut = "HARAM"
            raison = f"Secteur interdit : {sector}"
            r_dette, r_cash, r_creances, score, purification, rev_haram = (
                "N/A", "N/A", "N/A", 0, 0, "N/A")

        # ══════ NIVEAU 2 : Industrie spécifique ══════
        elif industry in INDUSTRIES_HARAM:
            statut = "HARAM"
            raison = f"Industrie interdite : {industry}"
            r_dette, r_cash, r_creances, score, purification, rev_haram = (
                "N/A", "N/A", "N/A", 0, 0, "N/A")

        # ══════ NIVEAU 3 : Liste noire manuelle ══════
        elif ticker in LISTE_NOIRE:
            statut = "HARAM"
            raison = LISTE_NOIRE[ticker]
            r_dette, r_cash, r_creances, score, purification, rev_haram = (
                "N/A", "N/A", "N/A", 0, 0, "N/A")

        else:
            # ══════ NIVEAU 4 : Revenus d'intérêts ══════
            if interest_income and total_revenue and total_revenue > 0:
                rev_haram = round(abs(interest_income) / abs(total_revenue) * 100, 2)
            else:
                rev_haram = 0

            # ══════ NIVEAU 5 : Ratios financiers AAOIFI ══════
            r_dette = round(total_debt / market_cap * 100, 1) if market_cap > 0 else 0
            r_cash = round(total_cash / market_cap * 100, 1) if market_cap > 0 else 0
            r_creances = round(receivables / market_cap * 100, 1) if market_cap > 0 else 0

            # ══════ NIVEAU 6 : Vérification zone grise ══════
            if industry in INDUSTRIES_ATTENTION:
                attention = INDUSTRIES_ATTENTION[industry]

            # ══════ VERDICT ══════
            if rev_haram > SEUIL_REV_HARAM:
                statut = "NON CONFORME"
                raison = f"Revenus haram : {rev_haram}% > {SEUIL_REV_HARAM}%"
            elif r_dette > SEUIL_DETTE:
                statut = "NON CONFORME"
                raison = f"Ratio dette : {r_dette}% > {SEUIL_DETTE}%"
            elif r_cash > SEUIL_CASH:
                statut = "NON CONFORME"
                raison = f"Ratio cash : {r_cash}% > {SEUIL_CASH}%"
            elif r_creances > SEUIL_CREANCES:
                statut = "NON CONFORME"
                raison = f"Ratio créances : {r_creances}% > {SEUIL_CREANCES}%"
            else:
                statut = "CONFORME"
                raison = "Tous les critères AAOIFI respectés"

            # Score
            score_dette = max(0, 100 - (r_dette / SEUIL_DETTE * 100))
            score_cash = max(0, 100 - (r_cash / SEUIL_CASH * 100))
            score_creances = max(0, 100 - (r_creances / SEUIL_CREANCES * 100))
            score_rev = max(0, 100 - (rev_haram / SEUIL_REV_HARAM * 100)) if rev_haram != "N/A" else 100
            score = round((score_dette + score_cash + score_creances + score_rev) / 4, 0)

            # Purification
            purification = rev_haram if isinstance(rev_haram, (int, float)) and rev_haram > 0 else 0

            # Si attention requise, baisser le score et signaler
            if attention:
                statut = statut + " ⚠️"
                score = max(0, score - 10)

        resultats_final.append({
            "Ticker": ticker,
            "Nom": nom,
            "Secteur": sector,
            "Industrie": industry,
            "Prix (€)": round(prix, 2) if prix else 0,
            "PER": round(per, 1) if isinstance(per, (int, float)) else "N/A",
            "Dividende (%)": round(dividend_yield * 100, 2) if dividend_yield else 0,
            "ROE (%)": round(roe * 100, 1) if roe else 0,
            "Marge nette (%)": round(marge * 100, 1) if marge else 0,
            "Croissance CA (%)": round(ca_growth * 100, 1) if ca_growth else 0,
            "Beta": round(beta, 2) if isinstance(beta, (int, float)) else "N/A",
            "Dette/Cap (%)": r_dette,
            "Cash/Cap (%)": r_cash,
            "Créances/Cap (%)": r_creances,
            "Rev. haram (%)": rev_haram,
            "Score Sharia": score,
            "Purification (%)": purification,
            "Statut": statut,
            "Raison": raison,
            "Attention": attention
        })

        emoji = "✅" if "CONFORME" in statut and "NON" not in statut else "⚠️" if "NON CONFORME" in statut else "❌"
        extra = " ⚠️ VÉRIFIER" if attention else ""
        print(f"{emoji} {nom} — {statut}{extra}")

    except Exception as e:
        print(f"❌ {nom}: {e}")

print(f"\n{'=' * 80}")
print(f"Analyse terminée — {len(resultats_final)} actions")
print(f"{'=' * 80}")

  HALAL QUANT V5 FINAL — SCREENING SHARIA COMPLET
--------------------------------------------------------------------------------
  Norme : AAOIFI
  Seuils : Dette < 33% | Cash < 33% | Créances < 49% | Revenus haram < 5%
  Filtres : Sectoriel + Industrie + Liste noire + Revenus d'intérêts + 3 ratios AAOIFI
✅ Air Liquide — CONFORME
✅ Airbus — CONFORME ⚠️ ⚠️ VÉRIFIER
⚠️ Schneider Electric — NON CONFORME
⚠️ ArcelorMittal — NON CONFORME
❌ AXA — HARAM
❌ BNP Paribas — HARAM
⚠️ Bouygues — NON CONFORME
✅ Capgemini — CONFORME
⚠️ Carrefour — NON CONFORME ⚠️ ⚠️ VÉRIFIER
❌ Crédit Agricole — HARAM
⚠️ Danone — NON CONFORME ⚠️ ⚠️ VÉRIFIER
✅ Dassault Systèmes — CONFORME
⚠️ Engie — NON CONFORME
✅ EssilorLuxottica — CONFORME
⚠️ Eurofins Scientific — NON CONFORME
✅ Hermès — CONFORME
❌ Kering — HARAM
✅ Legrand — CONFORME
✅ L'Oréal — CONFORME
❌ LVMH — HARAM
✅ Michelin — CONFORME
⚠️ Orange — NON CONFORME
❌ Pernod Ricard — HARAM
✅ Publicis — CONFORME
⚠️ Renault — NON CONFORME
✅ Safran — CONFORME ⚠️ ⚠️ VÉRIF

In [14]:
df_final = pd.DataFrame(resultats_final)

conf = df_final[df_final['Statut'].str.contains('CONFORME') & ~df_final['Statut'].str.contains('NON')].sort_values('Score Sharia', ascending=False)
non_conf = df_final[df_final['Statut'].str.contains('NON CONFORME')]
haram = df_final[df_final['Statut'].str.contains('HARAM')]

print("=" * 90)
print("  HALAL QUANT — SCREENING SHARIA FINAL (AAOIFI)")
print("=" * 90)

# Conformes
print(f"\n✅ CONFORMES ({len(conf)}):\n")
for _, row in conf.iterrows():
    warning = f" ⚠️ {row['Attention']}" if row['Attention'] else ""
    print(f"  {row['Nom']:.<25} Score: {row['Score Sharia']:>5} | "
          f"Prix: {row['Prix (€)']:>8}€ | PER: {str(row['PER']):>6} | "
          f"Div: {row['Dividende (%)']:>5}% | "
          f"Dette: {row['Dette/Cap (%)']:>5}% | "
          f"Purif: {row['Purification (%)']:>5}%{warning}")

# Non conformes
print(f"\n⚠️ NON CONFORMES ({len(non_conf)}):\n")
for _, row in non_conf.iterrows():
    print(f"  {row['Nom']:.<25} {row['Raison']}")

# Haram
print(f"\n❌ HARAM ({len(haram)}):\n")
for _, row in haram.iterrows():
    print(f"  {row['Nom']:.<25} {row['Raison']}")

# Résumé
total = len(df_final)
print(f"\n{'=' * 90}")
print(f"📊 RÉSUMÉ : {len(conf)} conformes | {len(non_conf)} non conformes | {len(haram)} haram")
print(f"📈 Taux de conformité : {round(len(conf)/total*100)}% du CAC 40")
print(f"⚠️  Actions conformes nécessitant vérification manuelle : "
      f"{len(conf[conf['Attention'] != ''])}")
print(f"{'=' * 90}")

  HALAL QUANT — SCREENING SHARIA FINAL (AAOIFI)

✅ CONFORMES (16):

  Capgemini................ Score: 100.0 | Prix:    109.5€ | PER:   12.0 | Div: 316.0% | Dette:   0.0% | Purif:     0%
  Michelin................. Score: 100.0 | Prix:    29.78€ | PER:    2.9 | Div:   0.0% | Dette:   0.0% | Purif:     0%
  Unibail-Rodamco.......... Score: 100.0 | Prix:      0.0€ | PER:    N/A | Div:   0.0% | Dette:     0% | Purif:     0%
  Hermès................... Score:  94.0 | Prix:   1908.0€ | PER:   44.4 | Div:  94.0% | Dette:   1.2% | Purif:     0%
  L'Oréal.................. Score:  91.0 | Prix:   361.25€ | PER:   31.6 | Div: 199.0% | Dette:   6.2% | Purif:     0%
  Schneider Electric....... Score:  87.0 | Prix:    252.2€ | PER:   31.7 | Div: 164.0% | Dette:  14.0% | Purif:     0%
  Air Liquide.............. Score:  86.0 | Prix:   169.16€ | PER:   27.7 | Div: 221.0% | Dette:  14.0% | Purif:     0%
  EssilorLuxottica......... Score:  86.0 | Prix:    209.5€ | PER:   42.1 | Div: 190.0% | Dette:  14

In [15]:
# ══════════════════════════════════════════════════════════════
# EXPORT EXCEL PROFESSIONNEL — HALAL QUANT V5
# ══════════════════════════════════════════════════════════════

from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.chart import PieChart, BarChart, Reference
from openpyxl.utils import get_column_letter

wb = Workbook()

# ══════════ ONGLET 1 : RÉSULTATS COMPLETS ══════════
ws1 = wb.active
ws1.title = "Screening CAC 40"
ws1.sheet_properties.tabColor = "1B2A4A"

# Titre
ws1.merge_cells('A1:T1')
ws1.cell(row=1, column=1).value = "HALAL QUANT — SCREENING SHARIA CAC 40 (Norme AAOIFI)"
ws1.cell(row=1, column=1).font = Font(bold=True, color="FFFFFF", name="Arial", size=14)
ws1.cell(row=1, column=1).fill = PatternFill("solid", fgColor="1B2A4A")
ws1.cell(row=1, column=1).alignment = Alignment(horizontal="center")
for c in range(1, 21):
    ws1.cell(row=1, column=c).fill = PatternFill("solid", fgColor="1B2A4A")

# En-têtes
headers = ['Ticker', 'Nom', 'Statut', 'Score Sharia', 'Raison', 'Prix (€)', 
           'PER', 'Dividende (%)', 'ROE (%)', 'Marge nette (%)', 'Croissance CA (%)',
           'Beta', 'Dette/Cap (%)', 'Cash/Cap (%)', 'Créances/Cap (%)',
           'Rev. haram (%)', 'Purification (%)', 'Secteur', 'Industrie', 'Attention']

for col, h in enumerate(headers, 1):
    cell = ws1.cell(row=2, column=col, value=h)
    cell.font = Font(bold=True, color="FFFFFF", name="Arial", size=9)
    cell.fill = PatternFill("solid", fgColor="2E75B6")
    cell.alignment = Alignment(horizontal="center", wrap_text=True)

# Couleurs
VERT = "E2F0E8"
ORANGE = "FDEBD0"
ROUGE = "FADBD8"
JAUNE = "FFF9C4"

# Trier les résultats
df_export = df_final.copy()
ordre = {"HARAM": 2, "NON CONFORME": 1}
df_export['tri'] = df_export['Statut'].apply(
    lambda x: 2 if 'HARAM' in x else 1 if 'NON CONFORME' in x else 0)
df_export = df_export.sort_values(['tri', 'Score Sharia'], ascending=[True, False])

# Remplir les données
for i, (_, row) in enumerate(df_export.iterrows(), 3):
    values = [row['Ticker'], row['Nom'], row['Statut'], row['Score Sharia'],
              row['Raison'], row['Prix (€)'], row['PER'], row['Dividende (%)'],
              row['ROE (%)'], row['Marge nette (%)'], row['Croissance CA (%)'],
              row['Beta'], row['Dette/Cap (%)'], row['Cash/Cap (%)'],
              row['Créances/Cap (%)'], row['Rev. haram (%)'],
              row['Purification (%)'], row['Secteur'], row['Industrie'], row['Attention']]
    
    statut = str(row['Statut'])
    if 'HARAM' in statut:
        couleur = ROUGE
    elif 'NON CONFORME' in statut:
        couleur = ORANGE
    elif row['Attention']:
        couleur = JAUNE
    else:
        couleur = VERT
    
    for col, val in enumerate(values, 1):
        cell = ws1.cell(row=i, column=col, value=val)
        cell.fill = PatternFill("solid", fgColor=couleur)
        cell.font = Font(name="Arial", size=9)
        cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)

# Largeurs de colonnes
widths = {'A': 10, 'B': 22, 'C': 20, 'D': 12, 'E': 40, 'F': 10, 'G': 8,
          'H': 12, 'I': 10, 'J': 12, 'K': 14, 'L': 8, 'M': 12, 'N': 12,
          'O': 14, 'P': 12, 'Q': 14, 'R': 20, 'S': 25, 'T': 40}
for col, width in widths.items():
    ws1.column_dimensions[col].width = width

# ══════════ ONGLET 2 : TOP ACTIONS HALAL ══════════
ws2 = wb.create_sheet("Top Actions Halal")
ws2.sheet_properties.tabColor = "2D8B4E"

ws2.merge_cells('A1:J1')
ws2.cell(row=1, column=1).value = "TOP ACTIONS HALAL — Classement par Score Sharia"
ws2.cell(row=1, column=1).font = Font(bold=True, color="FFFFFF", name="Arial", size=14)
ws2.cell(row=1, column=1).fill = PatternFill("solid", fgColor="2D8B4E")
for c in range(1, 11):
    ws2.cell(row=1, column=c).fill = PatternFill("solid", fgColor="2D8B4E")

headers2 = ['Rang', 'Nom', 'Score Sharia', 'Prix (€)', 'PER', 'Dividende (%)',
            'ROE (%)', 'Dette/Cap (%)', 'Purification (%)', 'Attention']
for col, h in enumerate(headers2, 1):
    cell = ws2.cell(row=2, column=col, value=h)
    cell.font = Font(bold=True, color="FFFFFF", name="Arial", size=10)
    cell.fill = PatternFill("solid", fgColor="1B2A4A")
    cell.alignment = Alignment(horizontal="center")

conf_sorted = df_final[
    df_final['Statut'].str.contains('CONFORME') & ~df_final['Statut'].str.contains('NON')
].sort_values('Score Sharia', ascending=False)

for i, (_, row) in enumerate(conf_sorted.iterrows(), 3):
    rang = i - 2
    values = [rang, row['Nom'], row['Score Sharia'], row['Prix (€)'],
              row['PER'], row['Dividende (%)'], row['ROE (%)'],
              row['Dette/Cap (%)'], row['Purification (%)'], row['Attention']]
    
    couleur = JAUNE if row['Attention'] else VERT
    for col, val in enumerate(values, 1):
        cell = ws2.cell(row=i, column=col, value=val)
        cell.fill = PatternFill("solid", fgColor=couleur)
        cell.font = Font(name="Arial", size=10, bold=(col <= 3))
        cell.alignment = Alignment(horizontal="center")

for col in ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J']:
    ws2.column_dimensions[col].width = 16
ws2.column_dimensions['B'].width = 24
ws2.column_dimensions['J'].width = 45

# ══════════ ONGLET 3 : GRAPHIQUES ══════════
ws3 = wb.create_sheet("Graphiques")
ws3.sheet_properties.tabColor = "D4A843"

# Données pour le camembert
ws3.cell(row=1, column=1).value = "Répartition"
ws3.cell(row=1, column=2).value = "Nombre"
ws3.cell(row=2, column=1).value = "Conformes"
ws3.cell(row=2, column=2).value = len(conf_sorted)
ws3.cell(row=3, column=1).value = "Non Conformes"
ws3.cell(row=3, column=2).value = len(df_final[df_final['Statut'].str.contains('NON CONFORME')])
ws3.cell(row=4, column=1).value = "Haram"
ws3.cell(row=4, column=2).value = len(df_final[df_final['Statut'].str.contains('HARAM')])

pie = PieChart()
pie.title = "Répartition Sharia du CAC 40"
pie.style = 10
data = Reference(ws3, min_col=2, min_row=1, max_row=4)
cats = Reference(ws3, min_col=1, min_row=2, max_row=4)
pie.add_data(data, titles_from_data=True)
pie.set_categories(cats)
pie.width = 18
pie.height = 12
ws3.add_chart(pie, "D1")

# Données pour le bar chart (scores)
ws3.cell(row=7, column=1).value = "Action"
ws3.cell(row=7, column=2).value = "Score Sharia"
for i, (_, row) in enumerate(conf_sorted.iterrows()):
    ws3.cell(row=8 + i, column=1).value = row['Nom']
    ws3.cell(row=8 + i, column=2).value = row['Score Sharia']

bar = BarChart()
bar.title = "Score Sharia — Actions Conformes du CAC 40"
bar.style = 10
bar.y_axis.title = "Score"
bar.x_axis.title = "Action"
data2 = Reference(ws3, min_col=2, min_row=7, max_row=7 + len(conf_sorted))
cats2 = Reference(ws3, min_col=1, min_row=8, max_row=7 + len(conf_sorted))
bar.add_data(data2, titles_from_data=True)
bar.set_categories(cats2)
bar.width = 28
bar.height = 14
ws3.add_chart(bar, "D18")

# ══════════ ONGLET 4 : MÉTHODOLOGIE ══════════
ws4 = wb.create_sheet("Méthodologie")
ws4.sheet_properties.tabColor = "7D3C98"

methodo = [
    ("HALAL QUANT — MÉTHODOLOGIE DE SCREENING", "", ""),
    ("", "", ""),
    ("Norme appliquée", "AAOIFI (Accounting and Auditing Organization for Islamic Financial Institutions)", ""),
    ("", "", ""),
    ("NIVEAU", "FILTRE", "DESCRIPTION"),
    ("Niveau 1", "Secteur principal", "Exclusion des secteurs Financial Services (banques, assurances)"),
    ("Niveau 2", "Industrie spécifique", "Exclusion de 25+ sous-industries (alcool, tabac, jeux, armes, crédit, etc.)"),
    ("Niveau 3", "Liste noire manuelle", "Entreprises avec revenus haram vérifiés (LVMH, Pernod Ricard, Kering)"),
    ("Niveau 4", "Revenus d'intérêts", "Revenus d'intérêts < 5% du chiffre d'affaires total"),
    ("Niveau 5", "Ratio dette AAOIFI", "Dette totale / Capitalisation boursière < 33%"),
    ("Niveau 5", "Ratio cash AAOIFI", "Cash et placements / Capitalisation boursière < 33%"),
    ("Niveau 5", "Ratio créances AAOIFI", "Créances nettes / Capitalisation boursière < 49%"),
    ("Niveau 6", "Zones grises", "Signalement des industries nécessitant vérification manuelle (défense, restauration, etc.)"),
    ("", "", ""),
    ("SCORE SHARIA", "Calcul", "Moyenne de 4 sous-scores : (100 - dette%) + (100 - cash%) + (100 - créances%) + (100 - rev.haram%). Pénalité de -10 si zone grise."),
    ("PURIFICATION", "Calcul", "% des dividendes à reverser en aumône = % de revenus d'intérêts de l'entreprise"),
    ("", "", ""),
    ("AVERTISSEMENT", "", "Ce screener est un outil d'aide à la décision. Il ne constitue pas une fatwa. Consultez un savant qualifié en fiqh al-mu'amalat pour toute décision d'investissement."),
]

for i, (a, b, c) in enumerate(methodo, 1):
    ws4.cell(row=i, column=1).value = a
    ws4.cell(row=i, column=2).value = b
    ws4.cell(row=i, column=3).value = c
    if i == 1:
        ws4.cell(row=i, column=1).font = Font(bold=True, size=14, color="1B2A4A")
    elif i == 5:
        for col in range(1, 4):
            ws4.cell(row=i, column=col).font = Font(bold=True, color="FFFFFF", size=10)
            ws4.cell(row=i, column=col).fill = PatternFill("solid", fgColor="1B2A4A")
    elif a.startswith("Niveau"):
        ws4.cell(row=i, column=1).font = Font(bold=True, color="2E75B6")
    elif a in ("SCORE SHARIA", "PURIFICATION", "AVERTISSEMENT"):
        ws4.cell(row=i, column=1).font = Font(bold=True, color="C0392B")

ws4.column_dimensions['A'].width = 20
ws4.column_dimensions['B'].width = 30
ws4.column_dimensions['C'].width = 80

# ══════════ SAUVEGARDER ══════════
wb.save('HALAL_QUANT_CAC40_V5.xlsx')
print("✅ Fichier Excel sauvegardé : HALAL_QUANT_CAC40_V5.xlsx")
print("   → Onglet 1 : Screening complet (40 actions)")
print("   → Onglet 2 : Top Actions Halal (classement)")
print("   → Onglet 3 : Graphiques (camembert + barres)")
print("   → Onglet 4 : Méthodologie (explication des filtres)")

✅ Fichier Excel sauvegardé : HALAL_QUANT_CAC40_V5.xlsx
   → Onglet 1 : Screening complet (40 actions)
   → Onglet 2 : Top Actions Halal (classement)
   → Onglet 3 : Graphiques (camembert + barres)
   → Onglet 4 : Méthodologie (explication des filtres)


In [16]:
!pip install openpyxl

zsh:1: command not found: pip


In [17]:
!pip3 install openpyxl


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip3 install --upgrade pip
